# Packages + Solver + Data Loading

In [3]:
using CSV, DataFrames, Statistics
using JuMP
using HiGHS
const SOLVER = HiGHS.Optimizer  

[ Info: Precompiling JuMP [4076af6c-e467-56ae-b986-b466b2749572] (cache misses: dep missing source (2))
[ Info: Precompiling HiGHS [87dc4568-4c63-4d18-b0c0-bb2238e4078b] (cache misses: wrong dep version loaded (2), wrong julia version (2))


HiGHS.Optimizer

In [5]:
grid = CSV.read("grid_pop.csv", DataFrame)
shelters_exist = CSV.read("shelter_final.csv", DataFrame)

Row,CS_ID,District,Upazila,union,lon,lat,Shelter Name (English),DRRO Capacity
,String7,String15,String7,String31,Float64,Float64,String,Float64
1,cs_1,cox's bazar,teknaf,teknaf paurashava,92.2907,20.8569,baitus saraf muhammodia riadul jannah dakil madrasha,1000.0
2,cs_10,cox's bazar,teknaf,sabrang,92.3322,20.7607,sahaporirdip dakkin para center,1000.0
3,cs_100,cox's bazar,ukhiya,palong khali,92.168,21.189,balokhali govt. primary school,400.0
4,cs_101,cox's bazar,ukhiya,palong khali,92.1681,21.1904,balukhali kamenia high school,250.0
5,cs_102,cox's bazar,ukhia,palong khali,92.1698,21.1448,farir beel alim madrasha,2500.0
6,cs_103,cox's bazar,ukhiya,palong khali,92.1615,21.1443,farir bill cyclone shelter cum gps,300.0
7,cs_104,cox's bazar,ukhia,palong khali,92.1615,21.1443,palongkhali govt. primary school,2500.0
8,cs_105,cox's bazar,ukhiya,palong khali,92.1667,21.1456,palongkhali model high school,250.0
9,cs_106,cox's bazar,ukhia,palong khali,92.1658,21.1633,rahamater bil govt. primary school,2500.0


In [6]:
# Column Mapping
# If we ever rename columns, we only need to edit here
# ---- Grid cells (I) ----
col_grid_id   = :grid_id
col_latI      = :lat_center
col_lonI      = :lon_center
col_pop       = :pop_cell
col_risk      = :hazard_score   

# ---- Existing shelters (J_E) ----
col_shel_id   = :CS_ID
col_latJE     = :lat
col_lonJE     = :lon
col_capJE     = Symbol("DRRO Capacity")  # space in column name

Symbol("DRRO Capacity")

# Distances

In [7]:
# Helper Functions
deg2rad(x) = x * (pi / 180)

"""
Haversine distance between two lat/lon points in kilometers.
"""
function haversine_km(lat1, lon1, lat2, lon2)
    R = 6371.0  # Earth radius in km
    φ1 = deg2rad(lat1); φ2 = deg2rad(lat2)
    Δφ = deg2rad(lat2 - lat1)
    Δλ = deg2rad(lon2 - lon1)
    a = sin(Δφ/2)^2 + cos(φ1)*cos(φ2)*sin(Δλ/2)^2
    c = 2 * atan(sqrt(a), sqrt(1-a))
    return R * c
end

haversine_km

# Build Sets + Parameters

In [8]:
# Build Sets + Parameters
# Population cells
I_ids = grid[!, col_grid_id]
latI  = grid[!, col_latI]
lonI  = grid[!, col_lonI]

# population d_i and risk r_i
d = coalesce.(grid[!, col_pop], 0.0)
r = coalesce.(grid[!, col_risk], 0.0)

nI = length(I_ids)

# totals for reporting
D_total = sum(d)
R_total = sum(r .* d)

# Existing shelters (J_E)
JE_ids = shelters_exist[!, col_shel_id]
latJE  = shelters_exist[!, col_latJE]
lonJE  = shelters_exist[!, col_lonJE]
uJE    = coalesce.(shelters_exist[!, col_capJE], 0.0)

nJE = length(JE_ids)
JE = 1:nJE  # indices for existing shelters

# Candidate shelters (J_N)
# every grid center, but: 
# - capacity = max existing capacity
# - remove candidates too close to existing shelters
u_new = maximum(uJE)   # assumption

# "on top of existing" filter threshold
eps_km = 0.804672 # 804.672 meters buffer/0.5 mi

keep_cand = trues(nI)
for i in 1:nI
    # distance from cell i to nearest existing shelter
    mind = minimum(haversine_km(latI[i], lonI[i], latJE[j], lonJE[j]) for j in 1:nJE)
    if mind < eps_km
        keep_cand[i] = false
    end
end

# build candidate arrays after filtering
JN_ids = ["cand_" * string(I_ids[i]) for i in 1:nI if keep_cand[i]]
latJN  = latI[keep_cand]
lonJN  = lonI[keep_cand]
uJN    = fill(u_new, length(JN_ids))

nJN = length(JN_ids)

# Combined shelter set J
# ----------------------------
J_ids = vcat(JE_ids, JN_ids)
latJ  = vcat(latJE, latJN)
lonJ  = vcat(lonJE, lonJN)
u     = vcat(uJE,  uJN)

nJ = length(J_ids)

# candidate indices within combined list
JN = (nJE+1):nJ

162:613

In [9]:
# Pre-compute Distance Matrix c_{ij}
c = Array{Float64}(undef, nI, nJ)
for i in 1:nI
    for j in 1:nJ
        c[i,j] = haversine_km(latI[i], lonI[i], latJ[j], lonJ[j])
    end
end

# Solver for a Single K

In [10]:
# Stage 1: maximize risk-weighted coverage
# Stage 2: minimize distance among max-coverage solutions
function solve_for_K(K::Int; delta_frac=1e-6)

    # Stage 1: Maximize risk-weighted covered population
    m1 = Model(SOLVER)
    set_silent(m1)

    @variable(m1, 0 <= x[1:nI, 1:nJ] <= 1)  # assignment fractions
    @variable(m1, y[1:nJ], Bin)             # shelter open decisions

    # Existing shelters always open
    for j in JE
        fix(y[j], 1.0; force=true)
    end

    # (1) each cell assigns at most 100% of its pop
    @constraint(m1, [i=1:nI], sum(x[i,j] for j in 1:nJ) <= 1)

    # (2) capacity constraints
    @constraint(m1, [j=1:nJ], sum(d[i]*x[i,j] for i in 1:nI) <= u[j]*y[j])

    # (3) at most K new shelters
    @constraint(m1, sum(y[j] for j in JN) <= K)

    # Risk-weighted coverage objective
    @objective(m1, Max, sum(r[i]*d[i]*x[i,j] for i in 1:nI, j in 1:nJ))

    optimize!(m1)
    if termination_status(m1) != MOI.OPTIMAL
        error("Stage 1 failed for K=$K")
    end

    Sr_star = objective_value(m1)  # maximum risk-weighted covered pop
    delta = delta_frac * R_total   # numerical tolerance for Stage 2

    # Stage 2: Minimize distance while preserving Sr_star
    m2 = Model(SOLVER)
    set_silent(m2)

    @variable(m2, 0 <= x2[1:nI, 1:nJ] <= 1)
    @variable(m2, y2[1:nJ], Bin)

    # Existing shelters fixed open again
    for j in JE
        fix(y2[j], 1.0; force=true)
    end

    # same feasibility constraints
    @constraint(m2, [i=1:nI], sum(x2[i,j] for j in 1:nJ) <= 1)
    @constraint(m2, [j=1:nJ], sum(d[i]*x2[i,j] for i in 1:nI) <= u[j]*y2[j])
    @constraint(m2, sum(y2[j] for j in JN) <= K)

    # preserve max risk coverage
    @constraint(m2,
        sum(r[i]*d[i]*x2[i,j] for i in 1:nI, j in 1:nJ) >= Sr_star - delta
    )

    # distance objective
    @objective(m2, Min, sum(c[i,j]*d[i]*x2[i,j] for i in 1:nI, j in 1:nJ))

    optimize!(m2)
    if termination_status(m2) != MOI.OPTIMAL
        error("Stage 2 failed for K=$K")
    end

    # Pull solution + compute reported stats
    x_sol = value.(x2)
    y_sol = value.(y2)

    Sr_realized = sum(r[i]*d[i]*x_sol[i,j] for i in 1:nI, j in 1:nJ)
    S_raw       = sum(d[i]*x_sol[i,j] for i in 1:nI, j in 1:nJ)

    alpha_r = R_total > 0 ? Sr_realized / R_total : 0.0
    alpha   = D_total > 0 ? S_raw / D_total       : 0.0
    Z_star  = objective_value(m2)

    return (alpha_r=alpha_r, alpha=alpha, distance=Z_star,
            Sr_star=Sr_star, y=y_sol, x=x_sol)
end

solve_for_K (generic function with 1 method)

In [ ]:
K_test = 50
sol = solve_for_K(K_test)

println(sol.alpha_r)   # risk-weighted coverage fraction
println(sol.alpha)     # raw population coverage fraction
println(sol.distance)  # total distance objective

In [11]:
# Save shelter decisions per K (for mapping)
for K in K_vals
    yK = ysols[K]
    shelter_out = DataFrame(
        shelter_id = J_ids,
        lat = latJ,
        lon = lonJ,
        capacity = u,
        is_new = [j in JN for j in 1:nJ],
        open = round.(Int, yK)
    )
    CSV.write("output.csv", shelter_out)
end

# Optional if we want to save assignments for a chosen K
function save_assignments(K::Int; thresh=1e-6)
    xK = xsols[K]
    rows = NamedTuple[]
    for i in 1:nI, j in 1:nJ
        if xK[i,j] > thresh
            push!(rows, (
                cell_id = I_ids[i],
                shelter_id = J_ids[j],
                frac_assigned = xK[i,j],
                pop_assigned = d[i]*xK[i,j],
                dist_km = c[i,j]
            ))
        end
    end
    assign_df = DataFrame(rows)
    CSV.write("assignments_K$(K).csv", assign_df)
    return assign_df
end

# Usage:
# save_assignments(2)

LoadError: UndefVarError: `K_vals` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [ ]:
K_test = 1
sol = solve_for_K(K_test)

println(sol.alpha_r)   # risk-weighted coverage fraction
println(sol.alpha)     # raw population coverage fraction
println(sol.distance)  # total distance objective